# Combined Anomaly Detection Analysis

This notebook combines prediction visualization and residual analysis for comprehensive anomaly detection insights.


## 1. Setup & Configuration


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
import re

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pio.templates.default = 'plotly_white'
%matplotlib inline

# Paths
DATA_DIR = Path('data')
PROCESSED_DIR = Path('..') / DATA_DIR / 'processed'
RESULTS_BASE_DIR = Path('..') / DATA_DIR / 'results'
MODELS_BASE_DIR = Path('..') / DATA_DIR / 'models'

# Configuration - Set to None to use latest
RESULTS_TIMESTAMP = None #"20260113_142936"  # e.g., '20251230_150706' or None
MODEL_NAME = 'LightGBM'
PANEL_ID = '1'  # Change to '2' for panel 2

print('Environment configured successfully!')


# 2. Download

In [ ]:
def find_latest_timestamped_dir(base_dir):
    """Find the most recent timestamped directory."""
    if not base_dir.exists():
        raise FileNotFoundError(f'Directory not found: {base_dir}')
    
    timestamp_pattern = re.compile(r'^\d{8}_\d{6}$')
    timestamped_dirs = [
        d for d in base_dir.iterdir() 
        if d.is_dir() and timestamp_pattern.match(d.name)
    ]
    
    if not timestamped_dirs:
        raise FileNotFoundError(f'No timestamped directories found in: {base_dir}')
    
    return sorted(timestamped_dirs, key=lambda d: d.name, reverse=True)[0]

# Get results directory
results_dir = (
    RESULTS_BASE_DIR / RESULTS_TIMESTAMP if RESULTS_TIMESTAMP 
    else find_latest_timestamped_dir(RESULTS_BASE_DIR)
)
print(f'Using results from: {results_dir.name}')


In [ ]:
# Load processed data
df_actual = pd.read_parquet(PROCESSED_DIR / f'panel_{PANEL_ID}.parquet')
if df_actual.index.name != 'timestamp':
    if 'timestamp' in df_actual.columns:
        df_actual = df_actual.set_index('timestamp')
df_actual.index = pd.to_datetime(df_actual.index)

# Load prediction results
df_forecast = pd.read_csv(results_dir / f'{MODEL_NAME}_panel_{PANEL_ID}_forecast.csv')
df_scores = pd.read_csv(results_dir / f'{MODEL_NAME}_panel_{PANEL_ID}_scores.csv')
df_flags = pd.read_csv(results_dir / f'{MODEL_NAME}_panel_{PANEL_ID}_flags.csv')

# Convert timestamps
for df in [df_forecast, df_scores, df_flags]:
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])

# Convert numeric columns (everything except timestamp) to numeric
for col in df_forecast.columns:
    if col != 'timestamp':
        df_forecast[col] = pd.to_numeric(df_forecast[col], errors='coerce')
        
for col in df_scores.columns:
    if col != 'timestamp':
        df_scores[col] = pd.to_numeric(df_scores[col], errors='coerce')
        
for col in df_flags.columns:
    if col != 'timestamp':
        df_flags[col] = pd.to_numeric(df_flags[col], errors='coerce')

# Load residuals from training (if available)
residuals_file = results_dir / f'{MODEL_NAME}_panel_{PANEL_ID}_residuals.csv'
df_residuals_train = None
if residuals_file.exists():
    df_residuals_train = pd.read_csv(residuals_file)
    if 'timestamp' in df_residuals_train.columns:
        df_residuals_train['timestamp'] = pd.to_datetime(df_residuals_train['timestamp'])
        df_residuals_train.set_index('timestamp', inplace=True)
    # Convert numeric columns
    for col in df_residuals_train.columns:
        df_residuals_train[col] = pd.to_numeric(df_residuals_train[col], errors='coerce')
    print(f'Training residuals loaded: {df_residuals_train.shape}')

print(f'\nData loaded for Panel {PANEL_ID}:')
print(f'  Actual data: {df_actual.shape}')
print(f'  Forecast: {df_forecast.shape}')
print(f'  Scores: {df_scores.shape}')
print(f'  Flags: {df_flags.shape}')
print(f'\nForecast columns: {list(df_forecast.columns)}')
print(f'Scores columns: {list(df_scores.columns)}')
print(f'Flags columns: {list(df_flags.columns)}')


In [ ]:
# Weather adjustment configuration
WEATHER_LATITUDE = 51.5
WEATHER_LONGITUDE = 10.5
WEATHER_LOOKBACK_DAYS = 14  # Last 2 weeks

# Import WeatherFetcher
from src.domain.weather_fetcher import WeatherFetcher

# Initialize WeatherFetcher
weather_config = {
    "latitude": WEATHER_LATITUDE,
    "longitude": WEATHER_LONGITUDE,
    "use_forecast": True,
}
weather_fetcher = WeatherFetcher(weather_config)

# Determine date range: last 2 weeks ending at the end of actual data
if len(df_actual) > 0:
    end_date = df_actual.index.max()
    start_date = end_date - pd.Timedelta(days=WEATHER_LOOKBACK_DAYS)
    
    print(f"\nFetching weather data for baseline calculation:")
    print(f"  Date range: {start_date} to {end_date}")
    
    try:
        # Fetch weather data
        weather_df = weather_fetcher.get_weather_data(
            start=start_date,
            end=end_date,
            timezone="UTC",
            fallback_on_failure=False,
        )
        
        if "temperature_2m" in weather_df.columns:
            # Calculate hourly profile: group by hour of day and compute mean
            weather_df["hour"] = weather_df.index.hour
            hourly_profile = weather_df.groupby("hour")["temperature_2m"].mean()
            
            print(f"  Successfully fetched {len(weather_df)} weather records")
            print(f"  Calculated hourly temperature profile (24 hours)")
            
            # Adjust temperature in actual data
            if "temperature" in df_actual.columns:
                df_actual_hours = pd.Series(df_actual.index.hour, index=df_actual.index)
                mean_temp = hourly_profile.mean()
                weather_baseline_actual = df_actual_hours.map(hourly_profile).fillna(mean_temp)
                df_actual["temperature"] = df_actual["temperature"] - weather_baseline_actual
                print(f"  Adjusted temperature in actual data")
            
            # Adjust temperature in forecast data
            if "temperature" in df_forecast.columns:
                # Get forecast timestamps (already converted to datetime in previous cell)
                if "timestamp" in df_forecast.columns:
                    forecast_timestamps = df_forecast["timestamp"]
                else:
                    # Use actual data timestamps if forecast doesn't have timestamp
                    forecast_timestamps = df_actual.index[-len(df_forecast):]
                
                forecast_hours = pd.Series(forecast_timestamps).dt.hour
                mean_temp = hourly_profile.mean()
                weather_baseline_forecast = forecast_hours.map(hourly_profile).fillna(mean_temp)
                df_forecast["temperature"] = df_forecast["temperature"] - weather_baseline_forecast.values
                print(f"  Adjusted temperature in forecast data")
            
            print("  Weather adjustment completed successfully!")
        else:
            print("  Warning: Weather data does not contain temperature_2m column")
            
    except Exception as e:
        print(f"  Warning: Failed to fetch weather data: {e}")
        print("  Skipping weather adjustment")
else:
    print("\nWarning: Actual data is empty, skipping weather adjustment")

## 3. Calculate Residuals & Merge Data


In [ ]:
# Check if forecast has timestamp column
if 'timestamp' in df_forecast.columns:
    # Set timestamp as index for alignment
    df_forecast_indexed = df_forecast.set_index('timestamp')
    df_scores_indexed = df_scores.set_index('timestamp') if 'timestamp' in df_scores.columns else df_scores
    df_flags_indexed = df_flags.set_index('timestamp') if 'timestamp' in df_flags.columns else df_flags
    
    # Align actual data to forecast timestamps
    df_actual_aligned = df_actual.loc[df_forecast_indexed.index]
else:
    # Forecast doesn't have timestamp - assume it corresponds to the last N rows of actual data
    print(f'Warning: Forecast CSV has no timestamp column. Using last {len(df_forecast)} rows of actual data.')
    df_actual_aligned = df_actual.iloc[-len(df_forecast):].copy()
    df_forecast_indexed = df_forecast.copy()
    df_forecast_indexed.index = df_actual_aligned.index
    df_scores_indexed = df_scores.copy()
    df_scores_indexed.index = df_actual_aligned.index
    df_flags_indexed = df_flags.copy()
    df_flags_indexed.index = df_actual_aligned.index

# Remove any unnamed columns from forecast
forecast_cols = [col for col in df_forecast_indexed.columns if not col.startswith('Unnamed') and col != 'timestamp']
df_forecast_clean = df_forecast_indexed[forecast_cols]

# Calculate residuals (actual - predicted)
df_residuals = df_actual_aligned[forecast_cols] - df_forecast_clean
df_abs_residuals = df_residuals.abs()
df_squared_residuals = df_residuals ** 2

# Create merged dataframe with suffixes
df_merged = df_actual_aligned.copy()
df_merged.columns = [f'{col}_actual' for col in df_merged.columns]

# Add forecast columns
for col in forecast_cols:
    df_merged[f'{col}_forecast'] = df_forecast_clean[col]

# Add scores and flags (handle both with and without 'timestamp' column)
score_col = [c for c in df_scores_indexed.columns if c != 'timestamp'][0] if len(df_scores_indexed.columns) > 0 else None
flag_col = [c for c in df_flags_indexed.columns if c != 'timestamp'][0] if len(df_flags_indexed.columns) > 0 else None

df_merged['anomaly_score'] = pd.to_numeric(df_scores_indexed[score_col], errors='coerce') if score_col else np.nan
df_merged['anomaly_flag'] = pd.to_numeric(df_flags_indexed[flag_col], errors='coerce').fillna(0).astype(int) if flag_col else 0

# Reset index for easier plotting
df_merged = df_merged.reset_index()

print(f'Merged data shape: {df_merged.shape}')
print(f'Residuals shape: {df_residuals.shape}')
print(f'\nAnomalies detected: {int(df_merged["anomaly_flag"].sum())}')
print(f'Anomaly rate: {(df_merged["anomaly_flag"].sum() / len(df_merged) * 100):.2f}%')


## 4. Overview Statistics


In [ ]:
print(f'=== Panel {PANEL_ID} Summary ===')
print(f'\nTime Range:')
print(f'  Start: {df_merged["timestamp"].min()}')
print(f'  End: {df_merged["timestamp"].max()}')
print(f'  Total points: {len(df_merged)}')

print(f'\nAnomaly Scores:')
print(f'  Mean: {df_merged["anomaly_score"].mean():.3f}')
print(f'  Std: {df_merged["anomaly_score"].std():.3f}')
print(f'  Min: {df_merged["anomaly_score"].min():.3f}')
print(f'  Max: {df_merged["anomaly_score"].max():.3f}')
print(f'  95th percentile: {df_merged["anomaly_score"].quantile(0.95):.3f}')

print(f'\nFeature Columns: {forecast_cols}')


## 5. Residual Analysis


### 5.1 Overall Feature Contribution


In [ ]:
# Calculate metrics per feature
mean_abs_residuals = df_abs_residuals.mean().sort_values(ascending=False)
rmse_per_feature = np.sqrt(df_squared_residuals.mean()).sort_values(ascending=False)

# For correlation, use the reset index versions to ensure alignment
df_abs_residuals_reset = df_abs_residuals.reset_index(drop=True)
correlations = df_abs_residuals_reset.corrwith(df_merged['anomaly_score']).sort_values(ascending=False)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mean absolute residuals
mean_abs_residuals.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_xlabel('Mean Absolute Residual')
axes[0].set_title('Average Prediction Error by Feature')
axes[0].grid(axis='x', alpha=0.3)

# RMSE
rmse_per_feature.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_xlabel('RMSE')
axes[1].set_title('Root Mean Squared Error by Feature')
axes[1].grid(axis='x', alpha=0.3)

# Correlation with anomaly score
correlations.plot(kind='barh', ax=axes[2], color='teal')
axes[2].set_xlabel('Correlation with Anomaly Score')
axes[2].set_title('Feature Residual Correlation with Score')
axes[2].axvline(x=0, color='black', linestyle='-', alpha=0.3)
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('\nTop 3 features by mean absolute residual:')
print(mean_abs_residuals.head(3))


### 5.2 High-Score Period Analysis


In [ ]:
# Define high-score threshold using the merged dataframe
high_score_threshold = df_merged['anomaly_score'].quantile(0.95)
high_score_mask = df_merged['anomaly_score'] > high_score_threshold

print(f'High score threshold (95th percentile): {high_score_threshold:.3f}')
print(f'Number of high-score timestamps: {high_score_mask.sum()}')

# Reset index on residuals to match merged dataframe
df_residuals_reset = df_residuals.reset_index(drop=True)
df_abs_residuals_reset = df_abs_residuals.reset_index(drop=True)

# Compare contributions
high_score_contributions = df_abs_residuals_reset[high_score_mask].mean().sort_values(ascending=False)
normal_score_contributions = df_abs_residuals_reset[~high_score_mask].mean().sort_values(ascending=False)
contribution_diff = (high_score_contributions - normal_score_contributions).sort_values(ascending=False)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

high_score_contributions.plot(kind='barh', ax=axes[0], color='crimson')
axes[0].set_xlabel('Mean Absolute Residual')
axes[0].set_title('Feature Contributions During High-Score Periods')
axes[0].grid(axis='x', alpha=0.3)

contribution_diff.plot(kind='barh', ax=axes[1], color='purple')
axes[1].set_xlabel('Residual Increase vs Normal Periods')
axes[1].set_title('What Makes High-Score Periods Different')
axes[1].grid(axis='x', alpha=0.3)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print('\nFeatures with highest increase during high-score periods:')
print(contribution_diff.head(3))


### 5.3 Score Contribution Estimation


In [ ]:
# Approximate per-feature score contribution (based on squared residuals)
df_squared_residuals_reset = df_squared_residuals.reset_index(drop=True)
total_squared_residuals = df_squared_residuals_reset.sum(axis=1)
feature_contributions_pct = df_squared_residuals_reset.div(total_squared_residuals, axis=0)
feature_score_contributions = feature_contributions_pct.mul(df_merged['anomaly_score'], axis=0)
avg_score_contribution = feature_score_contributions.mean().sort_values(ascending=False)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
avg_score_contribution.plot(kind='barh', ax=ax, color='darkgreen')
ax.set_xlabel('Average Score Contribution')
ax.set_title('Estimated Average Contribution to Anomaly Score by Feature')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nEstimated average score contributions:')
print(avg_score_contribution)


## 6. Interactive Visualizations


### 6.1 Anomaly Scores Over Time


In [ ]:
# Create interactive anomaly score plot
fig = go.Figure()

# Add anomaly score trace
fig.add_trace(go.Scatter(
    x=df_merged['timestamp'],
    y=df_merged['anomaly_score'],
    mode='lines',
    name='Anomaly Score',
    line=dict(color='black', width=2),
    fill='tozeroy',
    fillcolor='rgba(0, 0, 0, 0.2)'
))

# Add threshold line
fig.add_hline(
    y=high_score_threshold,
    line_dash='dash',
    line_color='red',
    annotation_text='95th Percentile',
    annotation_position='right'
)

# Highlight anomaly flags
anomaly_points = df_merged[df_merged['anomaly_flag'] == 1]
if len(anomaly_points) > 0:
    fig.add_trace(go.Scatter(
        x=anomaly_points['timestamp'],
        y=anomaly_points['anomaly_score'],
        mode='markers',
        name='Flagged Anomaly',
        marker=dict(color='red', size=10, symbol='circle')
    ))

fig.update_layout(
    title=f'Panel {PANEL_ID} - Anomaly Scores Over Time',
    xaxis_title='Timestamp',
    yaxis_title='Anomaly Score',
    height=500,
    hovermode='x unified'
)

fig.show()


### 6.2 All Channels Overview


In [ ]:
# Get actual columns
actual_cols = [col for col in df_merged.columns if col.endswith('_actual')]
n_channels = len(actual_cols)
n_cols = 2
n_rows = (n_channels + 1) // 2

# Create subplots
subplot_titles = [col.replace('_actual', '') for col in actual_cols]
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=subplot_titles,
    vertical_spacing=0.08,
    horizontal_spacing=0.08
)

# Plot each channel
for idx, col in enumerate(actual_cols):
    row = idx // n_cols + 1
    col_num = idx % n_cols + 1
    forecast_col = col.replace('_actual', '_forecast')
    
    # Actual
    fig.add_trace(
        go.Scatter(
            x=df_merged['timestamp'],
            y=df_merged[col],
            mode='lines',
            name='Actual',
            line=dict(color='blue', width=1.5),
            showlegend=(idx == 0),
            legendgroup='actual'
        ),
        row=row, col=col_num
    )
    
    # Forecast
    if forecast_col in df_merged.columns:
        fig.add_trace(
            go.Scatter(
                x=df_merged['timestamp'],
                y=df_merged[forecast_col],
                mode='lines',
                name='Forecast',
                line=dict(color='orange', width=1.5, dash='dash'),
                showlegend=(idx == 0),
                legendgroup='forecast'
            ),
            row=row, col=col_num
        )
    
    # Anomalies
    if len(anomaly_points) > 0:
        fig.add_trace(
            go.Scatter(
                x=anomaly_points['timestamp'],
                y=anomaly_points[col],
                mode='markers',
                name='Anomaly',
                marker=dict(color='red', size=5, symbol='circle'),
                showlegend=(idx == 0),
                legendgroup='anomaly'
            ),
            row=row, col=col_num
        )

fig.update_layout(
    height=400 * n_rows,
    title_text=f'Panel {PANEL_ID} - All Channels with Predictions',
    showlegend=True,
    hovermode='closest'
)

fig.show()


### 6.3 Top Contributing Features with Anomaly Scores


In [ ]:
# Get top 3 contributing features
top_features = mean_abs_residuals.head(3).index
n_features = len(top_features)

# Create subplots
fig = make_subplots(
    rows=n_features + 1, cols=1,
    subplot_titles=['Anomaly Scores'] + [f'{feat} - Absolute Residuals' for feat in top_features],
    vertical_spacing=0.06,
    row_heights=[1] + [0.8] * n_features
)

# Plot anomaly scores
fig.add_trace(
    go.Scatter(
        x=df_merged['timestamp'],
        y=df_merged['anomaly_score'],
        mode='lines',
        name='Anomaly Score',
        line=dict(color='black', width=2),
        fill='tozeroy'
    ),
    row=1, col=1
)

# Add threshold
fig.add_hline(
    y=high_score_threshold,
    line_dash='dash',
    line_color='red',
    row=1, col=1
)

# Plot top features' absolute residuals
for i, feature in enumerate(top_features, 2):
    fig.add_trace(
        go.Scatter(
            x=df_merged['timestamp'],
            y=df_abs_residuals.reset_index(drop=True)[feature],
            mode='lines',
            name=feature,
            line=dict(width=2),
            fill='tozeroy'
        ),
        row=i, col=1
    )

fig.update_layout(
    height=300 * (n_features + 1),
    title_text=f'Panel {PANEL_ID} - Anomaly Scores and Top Contributing Features',
    showlegend=True,
    hovermode='x unified'
)

fig.update_xaxes(title_text='Timestamp', row=n_features + 1, col=1)

fig.show()


## 7. Top Anomaly Instances


In [ ]:
# Get top 5 anomalies from merged dataframe
top_n = 5
top_anomaly_rows = df_merged.nlargest(top_n, 'anomaly_score')

print(f'Top {top_n} Anomaly Scores:')
print('=' * 80)

for idx, (row_idx, row) in enumerate(top_anomaly_rows.iterrows(), 1):
    score = row['anomaly_score']
    is_flagged = row['anomaly_flag'] == 1
    timestamp = row['timestamp']
    
    print(f'\n{idx}. Timestamp: {timestamp}')
    print(f'   Anomaly Score: {score:.4f}')
    print(f'   Is Flagged: {"Yes" if is_flagged else "No"}')
    print(f'\n   Top 3 Contributing Features:')
    
    # Get residuals for this row
    residuals_row = df_abs_residuals.iloc[row_idx]
    feature_residuals = residuals_row.sort_values(ascending=False)
    
    for feature, residual in feature_residuals.head(3).items():
        actual_col = f'{feature}_actual'
        forecast_col = f'{feature}_forecast'
        if actual_col in row.index and forecast_col in row.index:
            actual_val = row[actual_col]
            pred_val = row[forecast_col]
            print(f'      - {feature:30s}: residual={residual:8.4f} '
                  f'(actual={actual_val:8.4f}, pred={pred_val:8.4f})')
    print('-' * 80)


## 8. Training Residuals (if available)


In [ ]:
if df_residuals_train is not None:
    print('Training Residuals Statistics:')
    print(df_residuals_train.describe())
    
    # Plot distribution
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Histogram of residuals
    score_col = [col for col in df_residuals_train.columns if 'score' in col.lower()]
    if score_col:
        score_col = score_col[0]
        df_residuals_train[score_col].hist(bins=50, ax=axes[0], color='skyblue', edgecolor='black')
        axes[0].set_xlabel('Anomaly Score')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Training: Anomaly Score Distribution')
        axes[0].grid(alpha=0.3)
        
        # Compare with prediction scores
        axes[1].hist(df_residuals_train[score_col].dropna(), bins=50, alpha=0.6, 
                     label='Training', color='skyblue', density=True)
        axes[1].hist(df_merged['anomaly_score'].dropna(), bins=50, alpha=0.6,
                     label='Prediction', color='coral', density=True)
        axes[1].set_xlabel('Anomaly Score')
        axes[1].set_ylabel('Density')
        axes[1].set_title('Training vs Prediction Score Distribution')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('No training residuals file found.')


## 9. Summary Report


In [ ]:
print(f'\n{"=" * 80}')
print(f'SUMMARY REPORT - Panel {PANEL_ID}')
print(f'{"=" * 80}\n')

print('1. Data Overview:')
print(f'   - Time range: {df_merged["timestamp"].min()} to {df_merged["timestamp"].max()}')
print(f'   - Total predictions: {len(df_merged)}')
print(f'   - Anomalies flagged: {df_merged["anomaly_flag"].sum()} ({(df_merged["anomaly_flag"].sum() / len(df_merged) * 100):.2f}%)')

print('\n2. Top 3 Features by Prediction Error:')
for i, (feature, value) in enumerate(mean_abs_residuals.head(3).items(), 1):
    print(f'   {i}. {feature}: {value:.4f}')

print('\n3. Top 3 Features Correlated with Anomaly Scores:')
for i, (feature, value) in enumerate(correlations.head(3).items(), 1):
    print(f'   {i}. {feature}: {value:.4f}')

print('\n4. Top 3 Features Contributing to Anomaly Scores:')
for i, (feature, value) in enumerate(avg_score_contribution.head(3).items(), 1):
    print(f'   {i}. {feature}: {value:.4f}')

print('\n5. High-Score Period Insights:')
print(f'   - High-score threshold (95th percentile): {high_score_threshold:.3f}')
print(f'   - Number of high-score timestamps: {high_score_mask.sum()}')
print(f'   - Top feature during high-scores: {contribution_diff.index[0]} '
      f'(+{contribution_diff.iloc[0]:.4f} increase)')

print(f'\n{"=" * 80}')


## 10. Export Results (Optional)


In [ ]:
# Uncomment to export
# output_dir = results_dir / 'analysis'
# output_dir.mkdir(exist_ok=True)

# # Export summary
# summary_df = pd.DataFrame({
#     'mean_abs_residual': mean_abs_residuals,
#     'rmse': rmse_per_feature,
#     'correlation_with_score': correlations,
#     'avg_score_contribution': avg_score_contribution,
#     'high_score_contribution': high_score_contributions,
#     'contribution_diff': contribution_diff
# })
# summary_df.to_csv(output_dir / f'panel_{PANEL_ID}_combined_summary.csv')

# # Export merged data
# df_merged.to_csv(output_dir / f'panel_{PANEL_ID}_merged_data.csv', index=False)

# print(f'Results exported to: {output_dir}')

print('Export section ready (uncomment to use)')
